## trade 数据

In [3]:
import os
import polars as pl
date = ['2025-12-08']
trade_dir = '/root/autodl-tmp/ETHUSDT/trade'
filename = f'ETHUSDT{date[0]}.csv.gz'
filepath = os.path.join(trade_dir, filename)
trade_tick = pl.read_csv(filepath,n_rows=100000)
trade_tick.head()


timestamp,symbol,side,size,price,tickDirection,trdMatchID,grossValue,homeNotional,foreignNotional,RPI
f64,str,str,f64,f64,str,str,f64,f64,f64,i64
1.7652e9,"""ETHUSDT""","""Buy""",0.01,3058.08,"""ZeroPlusTick""","""f466fb98-92e4-5eac-9693-9b79a0…",3.0581e9,0.01,30.5808,0
1.7652e9,"""ETHUSDT""","""Sell""",0.92,3058.07,"""MinusTick""","""50b875d2-8c42-530f-a5c1-e057d5…",2.8134e11,0.92,2813.4244,0
1.7652e9,"""ETHUSDT""","""Sell""",0.05,3058.07,"""ZeroMinusTick""","""7038aaa4-50d9-55dc-b6e9-6ef430…",1.5290e10,0.05,152.9035,0
1.7652e9,"""ETHUSDT""","""Sell""",0.03,3058.07,"""ZeroMinusTick""","""9058425c-6fc2-5853-bec8-b68b36…",9.1742e9,0.03,91.7421,0
1.7652e9,"""ETHUSDT""","""Sell""",0.23,3058.07,"""ZeroMinusTick""","""a9259ada-1c85-5217-abb1-485502…",7.0336e10,0.23,703.3561,0


In [4]:
trade_tick

timestamp,symbol,side,size,price,tickDirection,trdMatchID,grossValue,homeNotional,foreignNotional,RPI
f64,str,str,f64,f64,str,str,f64,f64,f64,i64
1.7652e9,"""ETHUSDT""","""Buy""",0.01,3058.08,"""ZeroPlusTick""","""f466fb98-92e4-5eac-9693-9b79a0…",3.0581e9,0.01,30.5808,0
1.7652e9,"""ETHUSDT""","""Sell""",0.92,3058.07,"""MinusTick""","""50b875d2-8c42-530f-a5c1-e057d5…",2.8134e11,0.92,2813.4244,0
1.7652e9,"""ETHUSDT""","""Sell""",0.05,3058.07,"""ZeroMinusTick""","""7038aaa4-50d9-55dc-b6e9-6ef430…",1.5290e10,0.05,152.9035,0
1.7652e9,"""ETHUSDT""","""Sell""",0.03,3058.07,"""ZeroMinusTick""","""9058425c-6fc2-5853-bec8-b68b36…",9.1742e9,0.03,91.7421,0
1.7652e9,"""ETHUSDT""","""Sell""",0.23,3058.07,"""ZeroMinusTick""","""a9259ada-1c85-5217-abb1-485502…",7.0336e10,0.23,703.3561,0
…,…,…,…,…,…,…,…,…,…,…
1.7652e9,"""ETHUSDT""","""Buy""",0.03,3061.69,"""ZeroPlusTick""","""41b5352e-6943-5cd2-bf61-736e1b…",9.1851e9,0.03,91.8507,0
1.7652e9,"""ETHUSDT""","""Buy""",0.03,3061.69,"""ZeroPlusTick""","""186408a3-a3d3-5faa-af0e-4d7f99…",9.1851e9,0.03,91.8507,0
1.7652e9,"""ETHUSDT""","""Buy""",0.03,3061.69,"""ZeroPlusTick""","""119a889b-0807-511d-847d-678378…",9.1851e9,0.03,91.8507,0


In [1]:
import torch
import torch.nn.functional as F

# ----------------------
# 1. 构造一个超级简单的测试数据
# ----------------------
# 形状：(B=1, C=1, T=5, F=1) → 批次1，通道1，时间长度5，特征1
x = torch.tensor([[[
    [1.0],   # T=0
    [2.0],   # T=1
    [3.0],   # T=2
    [4.0],   # T=3
    [5.0]    # T=4
]]])

print("【原始输入】shape:", x.shape)
print(x.squeeze().numpy())  # 打印成一维方便看

# ----------------------
# 2. 因果填充：左侧 pad=2 (kernel=3 需要 pad 2个)
# F.pad 参数：(左, 右, 上, 下)
# ----------------------
time_pad = 2  # 时间维度左侧填充 2 个

# 🔥 关键：使用 replicate 模式
x_padded = F.pad(x, (0, 0, time_pad, 0), mode="replicate")

print("\n【Replicate 填充后】shape:", x_padded.shape)
print(x_padded.squeeze().numpy())


libgomp: Invalid value for environment variable OMP_NUM_THREADS


【原始输入】shape: torch.Size([1, 1, 5, 1])
[1. 2. 3. 4. 5.]

【Replicate 填充后】shape: torch.Size([1, 1, 7, 1])
[1. 1. 1. 2. 3. 4. 5.]


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================
# Utility
# =========================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


# =========================
# [1] Level-wise Encoder
# =========================

class LevelEncoder(nn.Module):
    def __init__(self, in_dim, d_model):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

    def forward(self, x, rel_price):
        # x: [B, T, L, F]
        # rel_price: [B, T, L, 1]
        x = torch.cat([x, rel_price], dim=-1)
        return self.mlp(x)  # [B, T, L, D]


# =========================
# [2] Side-aware Cross Attention
# =========================
## 显式的建模ask和bid的博弈
class SideCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)

    def forward(self, bid, ask):
        # bid/ask: [B, T, L/2, D]
        B, T, L, D = bid.shape

        bid = bid.reshape(B*T, L, D)
        ask = ask.reshape(B*T, L, D)

        bid2, _ = self.attn(bid, ask, ask)
        ask2, _ = self.attn(ask, bid, bid)

        return bid2.reshape(B, T, L, D), ask2.reshape(B, T, L, D)


# =========================
# [3] Temporal Patch Pooling
# =========================

class PatchPooling(nn.Module):
    def __init__(self, d_model, patch_size):
        super().__init__()
        self.patch_size = patch_size
        self.attn = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: [B, T, D]
        B, T, D = x.shape
        P = self.patch_size

        T_new = T // P
        x = x[:, :T_new * P]
        x = x.reshape(B, T_new, P, D)

        weights = torch.softmax(self.attn(x), dim=2)  # [B, T', P, 1]
        x = (x * weights).sum(dim=2)

        return x  # [B, T', D]


# =========================
# [4] Change-aware Transformer
# =========================

class ChangeAwareAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.linear = nn.Linear(d_model * 2, d_model)

    def forward(self, x):
        # x: [B, T, D]
        dx = torch.zeros_like(x)
        dx[:, 1:] = x[:, 1:] - x[:, :-1]

        x_aug = torch.cat([x, dx], dim=-1)
        x_aug = self.linear(x_aug)

        out, _ = self.attn(x_aug, x_aug, x_aug)
        return out


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()
        self.attn = ChangeAwareAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(x)
        x = self.norm1(x)
        x = x + self.ff(x)
        x = self.norm2(x)
        return x


# =========================
# Full Model
# =========================

class MAHT(nn.Module):
    def __init__(self, 
                 in_dim=2,   # price, volume
                 d_model=64,
                 n_heads=4,
                 num_layers=3,
                 patch_size=10,
                 horizon=3):
        super().__init__()

        self.level_encoder = LevelEncoder(in_dim + 1, d_model)

        self.side_attn = SideCrossAttention(d_model, n_heads)

        self.patch_pool = PatchPooling(d_model, patch_size)

        self.pos_enc = PositionalEncoding(d_model)

        self.transformer = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_model * 4)
            for _ in range(num_layers)
        ])

        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        # x: [B, T, L, 2]  (price, volume)

        B, T, L, F = x.shape

        mid = x[:, :, L//2, 0:1]
        rel_price = (x[..., 0:1] - mid.unsqueeze(2))

        h = self.level_encoder(x, rel_price)

        # split bid / ask
        bid = h[:, :, :L//2]
        ask = h[:, :, L//2:]

        bid, ask = self.side_attn(bid, ask)

        h = torch.cat([bid, ask], dim=2)

        # pool over levels
        h = h.mean(dim=2)  # [B, T, D]

        # temporal patch
        h = self.patch_pool(h)

        h = self.pos_enc(h)

        for layer in self.transformer:
            h = layer(h)

        out = self.head(h[:, -1])  # last token

        return out


# =========================
# Quick test
# =========================

if __name__ == "__main__":
    model = MAHT()

    x = torch.randn(8, 300, 20, 2)  # batch, time, levels, features
    y = model(x)

    print("Output shape:", y.shape)  # [8, horizon]

In [ ]:
import torch
import torch.nn as nn

class MicroStructureLayer(nn.Module):
    """
    注入隐式 microstructure bias 的第一层。
    它负责将原始 (T, 20, 2) 张量转化为 Transformer Token 序列。
    """
    def __init__(self, channels=2, num_levels=20, embed_dim=128):
        super().__init__()
        # Bias Point 1: LOB-Conv 捕捉档位互动
        # 使用 Depthwise Separable Conv 压缩 Channels
        self.lob_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3, channels), stride=(1, channels), padding=(1, 0)),
            nn.ReLU(),
            nn.Conv2d(16, embed_dim, kernel_size=(3, 1), stride=(2, 1), padding=(1, 0)), # 20->10
            nn.ReLU()
        )
        
        # 将空间维度 (10) 拍平并投影为 Token
        self.folding = nn.Sequential(
            nn.Flatten(2), # (T, 10 * embed_dim)
            nn.Linear(10 * embed_dim, embed_dim),
            nn.Tanh() # 限制极值，增强降噪
        )

    def forward(self, x):
        # x shape: (B, T, 20, C)
        B, T, L, C = x.shape
        
        # Preprocessing: Relative Pricing Bias (隐式 Micro-price)
        mid_price = (x[:, :, L//2, 0] + x[:, :, L//2-1, 0]) / 2.0
        x_norm = x.clone()
        x_norm[:, :, :, 0] = x_norm[:, :, :, 0] - mid_price.unsqueeze(2)
        
        # Prepare for Conv2d: (B, 1, T, L*C)
        x_norm = x_norm.view(B, 1, T, L * C)
        
        # Bias Point 2: Folding (时空折叠)
        # 这一步将 (T, 20, 2) 变为维度为 (T, D) 的高频语义序列
        feature_map = self.lob_conv(x_norm) # (B, embed_dim, T_reduced, 1)
        feature_sequence = feature_map.squeeze(-1).permute(0, 2, 1) # (B, T_reduced, embed_dim)
        
        token_sequence = self.folding(feature_sequence) # (B, T_reduced, embed_dim)
        
        return token_sequence

class PatchLOBTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.micro_layer = MicroStructureLayer(embed_dim=128)
        
        # Bias Point 3: Patching & Instance Norm
        self.patcher = nn.Unfold(kernel_size=(20, 128), stride=(10, 128)) # P=20, S=10
        self.token_embed = nn.Linear(20 * 128, 256)
        
        # 骨干网络使用 PatchTST 的 Transformer 层
        self.transformer = PatchTSTEncoder(num_patch=29, d_model=256) # 假设 T=3000
        self.head = nn.Linear(256, 1) # 预测收益率

    def forward(self, x):
        # x shape: (B, Time, 20, Channels)
        
        # 1. 获得具有 microstructure bias 的序列
        micro_tokens = self.micro_layer(x) # (B, T_reduced, 128)
        
        # 2. Patching (统计稳定性 Bias)
        B, T_r, D = micro_tokens.shape
        # Instance Norm on sequence
        micro_tokens_norm = (micro_tokens - micro_tokens.mean(1, keepdim=True)) / \
                            (micro_tokens.std(1, keepdim=True) + 1e-5)
        
        # Reshape for Unfold: (B, 1, T_r, D)
        patches = self.patcher(micro_tokens_norm.unsqueeze(1)) # (B, N_patches, P*D)
        
        patch_tokens = self.token_embed(patches.permute(0, 2, 1)) # (B, N_patches, 256)
        
        # 3. Transformer & Prediction
        output = self.transformer(patch_tokens) # (B, N_patches, 256)
        output = output[:, -1, :] # 取最后一个 Patch 作为总结
        
        prediction = self.head(output)
        return prediction